In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('creditcard.csv')
print(df.head())
print(df.info())
print(df['Class'].value_counts())



# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train-Test Split (80-20), stratified to maintain class imbalance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Fraud in training:", y_train.sum())
print("Fraud in testing:", y_test.sum())


   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [5]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE only to training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print("Training samples:", X_train_resampled.shape[0])
print("Fraud cases:", sum(y_train_resampled))
print("Legit cases:", len(y_train_resampled) - sum(y_train_resampled))


After SMOTE:
Training samples: 454902
Fraud cases: 227451
Legit cases: 227451


In [6]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit only on training data (resampled) and transform both train & test
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# Optional: Save scaler for deployment later
import joblib
joblib.dump(scaler, 'scaler.pkl')


['scaler.pkl']

In [8]:
from xgboost import XGBClassifier

# Initialize and train the XGBoost model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train_resampled)

# Optional: Save model
joblib.dump(xgb_model, 'fraud_xgb_model.pkl')
print("Model trained and saved as 'fraud_xgb_model.pkl'")


e:\prasunet\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:06:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model trained and saved as 'fraud_xgb_model.pkl'


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predict using the test set
y_pred = xgb_model.predict(X_test_scaled)
y_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")


Confusion Matrix:
[[56844    20]
 [   14    84]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9996    0.9997     56864
           1     0.8077    0.8571    0.8317        98

    accuracy                         0.9994     56962
   macro avg     0.9037    0.9284    0.9157     56962
weighted avg     0.9994    0.9994    0.9994     56962


ROC-AUC Score: 0.9854


In [1]:


# New cell for model comparison
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import numpy as np
import time

# Dictionary to store results
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}

for name, model in models.items():
    # Start timing
    start_time = time.time()
    
    # Fit model
    print(f"\nTraining {name}...")
    model.fit(X_train_scaled, y_train_resampled)
    
    # Training time
    train_time = time.time() - start_time
    
    # Predict and evaluate
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    # Calculate average precision score
    average_precision = average_precision_score(y_test, y_proba)
    
    # Store results
    results[name] = {
        'model': model,
        'confusion_matrix': cm,
        'classification_report': report,
        'roc_auc': roc_auc,
        'average_precision': average_precision,
        'training_time': train_time,
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    
    # Print results
    print(f"\n{name} Results:")
    print(f"Training Time: {train_time:.2f} seconds")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"Average Precision Score: {average_precision:.4f}")

# Save the best model (based on ROC-AUC)
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']
print(f"\nBest model based on ROC-AUC: {best_model_name}")

# Save the best model
joblib.dump(best_model, 'best_fraud_model.pkl')
print(f"Best model saved as 'best_fraud_model.pkl'")

NameError: name 'XGBClassifier' is not defined